In [ ]:
클릭된 광고 예측

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# ================================
# 1. 데이터 불러오기
# ================================
df = pd.read_csv("./data/Clicked Ads Dataset.csv")

print("컬럼 목록:", df.columns)

# ================================
# 2. X(입력), y(라벨) 분리
# ================================

le = LabelEncoder()
y = le.fit_transform(df["Clicked on Ad"])   # 클릭 여부가 타겟

# 숫자형 컬럼만 선택 (문자열 칼럼 제거)
X = df.select_dtypes(include=["int64", "float64"])

# ================================
# 3. 데이터 분할 & 스케일링
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ================================
# 4. 모델 구성
# ================================
model = Sequential()
model.add(Dense(32, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid')) # 이진 분류

# ================================
# 5. 컴파일 & 학습
# ================================
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.25,
    verbose=1
)

# ================================
# 6. 평가
# ================================
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"테스트 정확도: {acc:.4f}")


In [ ]:
# 예측할 샘플 데이터 (DataFrame 형태)
new_data = pd.DataFrame({
    "Unnamed: 0": [999],  # 학습 때 있던 컬럼 dummy로 추가
    "Daily Time Spent on Site": [68.5],
    "Age": [35],
    "Area Income": [65000],
    "Daily Internet Usage": [200],
    # "Male" 컬럼은 학습 데이터에 없으면 제거
})

# 1. 스케일링
new_data_scaled = scaler.transform(new_data)

# 2. 예측 (확률)
y_prob = model.predict(new_data_scaled)
print("클릭 확률:", y_prob[0][0])

# 3. 0/1 클래스로 변환
y_pred_class = (y_prob > 0.5).astype(int)
print("예측 클래스:", y_pred_class[0][0])

# 4. LabelEncoder 역변환 (원래 레이블)
y_pred_label = le.inverse_transform(y_pred_class)
print("예측 라벨:", y_pred_label[0])

